# Milestone 4 - reading your own practice history

**Why this matters.** This is the aggregate-and-rank shape that turns up
constantly in interviews: group records by a key, compute a rate per group, pick
the extreme. The twist is the tie-break, which is where most first drafts are
subtly wrong.

**New to coding?** Do `m0_python_basics.ipynb` first. Words you'll meet here: aggregation, dict, sorting with a tie-break. Each one is explained in plain language in [docs/CONCEPTS.md](../docs/CONCEPTS.md), along with Big-O and a routine for getting started when the cell is blank.

**How to work through this notebook.** Run the cells top to bottom. Each
exercise has a cell for you to fill in, then a cell of tests. Run the tests
first and watch them fail - read *what* they check, because the tests are the
specification. Then write code until they go green.

After you edit an exercise cell, **re-run it** before re-running its tests - the
tests use whatever is currently defined in the notebook.

In [ ]:
import ipytest
import pytest

# clean="test_*": before each test cell, forget test functions from earlier cells
# - but not other names starting with "Test", like milestone 2's TestCase class.
ipytest.autoconfig(clean="test_*")

from dataclasses import dataclass


@dataclass
class AttemptRecord:
    """One finished session."""

    problem_id: str
    topic: str
    solved: bool
    attempts: int


def rec(problem_id: str, topic: str, solved: bool) -> AttemptRecord:
    return AttemptRecord(problem_id=problem_id, topic=topic, solved=solved, attempts=1)

## 1. `stats_by_topic`

Aggregate a history into per-topic counts, shaped like:

```python
{"hash-map": {"attempted": 3, "solved": 2}, "stack": {"attempted": 1, "solved": 0}}
```

An empty history returns an empty dict - not a dict of zeros.

In [ ]:
def stats_by_topic(history: list[AttemptRecord]) -> dict[str, dict[str, int]]:
    """Aggregate a history into per-topic counts."""
    # TODO: implement.
    raise NotImplementedError("Milestone 4: implement stats_by_topic")

In [ ]:
%%ipytest -qq

def test_empty_history_gives_empty_stats():
    assert stats_by_topic([]) == {}


def test_counts_attempts_and_solves():
    history = [rec("a", "hash-map", True), rec("b", "hash-map", False)]
    assert stats_by_topic(history) == {"hash-map": {"attempted": 2, "solved": 1}}


def test_topics_are_counted_separately():
    stats = stats_by_topic([rec("a", "hash-map", True), rec("b", "stack", False)])
    assert stats["hash-map"]["solved"] == 1
    assert stats["stack"]["solved"] == 0

## 2. `weakest_topic`

Return the topic you are worst at, or `None` if the history is empty. "Worst"
means the lowest solved/attempted ratio.

The tie-break is the interesting part: when two topics have the same ratio,
prefer the one you have practised **less**. Without that rule, a topic you tried
once and failed ranks equal with one you failed ten times, and selection can get
stuck on the well-explored one. Sorting by a tuple gets you both rules at once -
think about what that tuple contains.

In [ ]:
def weakest_topic(history: list[AttemptRecord]) -> str | None:
    """Return the topic you are worst at, or None if the history is empty."""
    # TODO: implement.
    raise NotImplementedError("Milestone 4: implement weakest_topic")

In [ ]:
%%ipytest -qq

def test_weakest_topic_of_an_empty_history_is_none():
    assert weakest_topic([]) is None


def test_weakest_topic_is_the_lowest_solve_rate():
    history = [rec("a", "hash-map", True), rec("b", "stack", False)]
    assert weakest_topic(history) == "stack"


def test_weakest_topic_compares_rates_not_totals():
    """stack is 1/2; hash-map is 2/3. Counting raw solves would pick the wrong one."""
    history = [
        rec("a", "hash-map", True), rec("b", "hash-map", True), rec("c", "hash-map", False),
        rec("d", "stack", True), rec("e", "stack", False),
    ]
    assert weakest_topic(history) == "stack"


def test_tie_breaks_toward_the_least_practised():
    """Both are 0%. The one you have barely touched is the one to work on."""
    history = [rec("a", "hash-map", False), rec("b", "hash-map", False),
               rec("c", "stack", False)]
    assert weakest_topic(history) == "stack"

## 3. `unsolved_ids`

Return the ids from `all_ids` that have never been solved. A problem counts as
solved if **any** record for it has `solved=True` - failing it twice and then
solving it means you solved it. Preserve the order of `all_ids`.

In [ ]:
def unsolved_ids(history: list[AttemptRecord], all_ids: list[str]) -> list[str]:
    """Return the ids from `all_ids` that have never been solved."""
    # TODO: implement.
    raise NotImplementedError("Milestone 4: implement unsolved_ids")

In [ ]:
%%ipytest -qq

def test_unsolved_excludes_anything_ever_solved():
    history = [rec("a", "hash-map", False), rec("a", "hash-map", True)]
    assert unsolved_ids(history, ["a", "b"]) == ["b"]


def test_unsolved_preserves_input_order():
    assert unsolved_ids([], ["c", "a", "b"]) == ["c", "a", "b"]


def test_unsolved_of_everything_solved_is_empty():
    assert unsolved_ids([rec("a", "t", True)], ["a"]) == []

## Try it on your real history

If you have done a few `uv run drill start` sessions, your progress file has real
records in it. This reads it with the finished tutor's store and runs *your*
functions over it.

In [ ]:
from drill.config import DrillSettings
from drill.progress import ProgressStore

store = ProgressStore(DrillSettings.from_env().progress_path)
mine = [AttemptRecord(r.problem_id, r.topic, r.solved, r.attempts) for r in store.load()]
print(f"{len(mine)} sessions")
print(stats_by_topic(mine))
print("weakest:", weakest_topic(mine))